# 🚀 Chapter 10-01b: Deploy a Full-Stack LLM App

---

## 📖 The Mission

Build and deploy a production-ready chatbot with:
- **FastAPI** backend with streaming
- **Caching** for cost optimization
- **LangSmith** tracing
- **Streamlit** frontend

---

## 🧪 Step 1: FastAPI Backend

In [ ]:
# !pip install fastapi uvicorn langchain-openai langchain-core python-dotenv streamlit

In [ ]:
# Here's the complete FastAPI backend (save as app.py)

BACKEND_CODE = '''
import os
from fastapi import FastAPI
from fastapi.responses import StreamingResponse
from pydantic import BaseModel
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache

load_dotenv()

# Enable LangSmith tracing
os.environ.setdefault("LANGCHAIN_TRACING_V2", "true")

# Enable caching
set_llm_cache(InMemoryCache())

app = FastAPI(title="LLM Chat API")
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0.7, streaming=True)

SYSTEM_PROMPT = "You are a helpful AI assistant. Be concise and clear."

class ChatRequest(BaseModel):
    message: str

@app.post("/chat")
async def chat(req: ChatRequest):
    response = llm.invoke([
        SystemMessage(content=SYSTEM_PROMPT),
        HumanMessage(content=req.message)
    ])
    return {"response": response.content}

@app.post("/chat/stream")
async def chat_stream(req: ChatRequest):
    async def generate():
        async for chunk in llm.astream([
            SystemMessage(content=SYSTEM_PROMPT),
            HumanMessage(content=req.message)
        ]):
            if chunk.content:
                yield chunk.content
    return StreamingResponse(generate(), media_type="text/plain")

@app.get("/health")
async def health():
    return {"status": "healthy"}

# Run: uvicorn app:app --reload --port 8000
'''

print("📄 FastAPI Backend (save as app.py):")
print(BACKEND_CODE)

## 🧪 Step 2: Streamlit Frontend

In [ ]:
FRONTEND_CODE = '''
import streamlit as st
import requests

st.title("🤖 AI Chat Assistant")
st.caption("Powered by GPT-4o-mini")

API_URL = "http://localhost:8000"

# Chat history
if "messages" not in st.session_state:
    st.session_state.messages = []

# Display chat history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Chat input
if prompt := st.chat_input("Ask anything..."):
    # Show user message
    st.session_state.messages.append({"role": "user", "content": prompt})
    with st.chat_message("user"):
        st.write(prompt)
    
    # Get response
    with st.chat_message("assistant"):
        with st.spinner("Thinking..."):
            try:
                resp = requests.post(f"{API_URL}/chat", json={"message": prompt})
                answer = resp.json()["response"]
            except Exception as e:
                answer = f"Error: {str(e)}"
        st.write(answer)
    st.session_state.messages.append({"role": "assistant", "content": answer})

# Run: streamlit run frontend.py
'''

print("📄 Streamlit Frontend (save as frontend.py):")
print(FRONTEND_CODE)

## 🧪 Step 3: Test the API Directly

In [ ]:
# Test without running the server (direct LangChain call)
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage
from langchain.cache import InMemoryCache
from langchain.globals import set_llm_cache
import time

load_dotenv()
set_llm_cache(InMemoryCache())
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

messages = [
    SystemMessage(content="You are a helpful assistant. Be concise."),
    HumanMessage(content="What is LangGraph in one sentence?")
]

# First call (uncached)
start = time.time()
response1 = llm.invoke(messages)
t1 = time.time() - start
print(f"First call ({t1:.2f}s): {response1.content}")

# Second call (cached!)
start = time.time()
response2 = llm.invoke(messages)
t2 = time.time() - start
print(f"Cached call ({t2:.4f}s): {response2.content}")
print(f"\n⚡ Cache speedup: {t1/max(t2, 0.001):.0f}x faster!")

## 🧪 Step 4: Docker Deployment

In [ ]:
DOCKERFILE = '''
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY . .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
'''

print("📄 Dockerfile:")
print(DOCKERFILE)
print("\n🐳 Build & Run:")
print("  docker build -t llm-chat .")
print("  docker run -p 8000:8000 --env-file .env llm-chat")

## ✅ Key Takeaways

```
┌─────────────────────────────────────────────────────────────────────┐
│  📌 FastAPI + LangChain = production-ready LLM API.                │
├─────────────────────────────────────────────────────────────────────┤
│  📌 StreamingResponse enables token-by-token output.               │
├─────────────────────────────────────────────────────────────────────┤
│  📌 InMemoryCache → instant responses for repeated queries.        │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Streamlit = fastest way to build chat UIs.                     │
├─────────────────────────────────────────────────────────────────────┤
│  📌 Docker → consistent deployment anywhere.                       │
└─────────────────────────────────────────────────────────────────────┘
```

---

## ➡️ Next Up: Guardrails & Safety!

---

### 🎉 You Can Deploy LLM Apps!
Backend + frontend + caching + Docker — production stack complete. 🚀